# Full activation-vector experiments: SAE

This companion extends `sae_gpt_small_tokens_places.ipynb`.
The original notebook is preserved. These experiments keep every
coordinate and amplitude of the source token codes: no top-feature
selection, minimum-positive replacement, or atomic reduction.

Six source occurrences, two query operations, and four relative
activation levels are fixed in the shared implementation. Layers
0, 8, and 11 run sequentially. The Rio/de pair adds one meet family.
All 25,600 cached rows contribute to every extent and statistic.
Models, tokens, and activation matrices are loaded offline.

In [1]:
import os
import sys
from pathlib import Path

os.environ['HF_HUB_OFFLINE'] = '1'
os.environ['TRANSFORMERS_OFFLINE'] = '1'
root = next(
    p for p in [Path.cwd(), *Path.cwd().parents]
    if (p / 'pyproject.toml').is_file()
)
paper = root / 'texs/sparsesurrs/lattcontexts'
sys.path.insert(0, str(paper / 'python'))
from lattcontexts_tools.activation_experiments_codexgen import (
    ALPHAS, CASES, OUT, run,
)
import pandas as pd

## Construction and interpretation

A row represents one 128-token corpus block. Its vector is the
coordinatewise maximum of all token activations in that block.
For selected source positions, the full-vector meet keeps the
smallest shared amplitude in each coordinate; the full-vector join
keeps the largest amplitude at any selected position.

At each level alpha in {0.25, 0.5, 0.75, 1}, a row qualifies if
its summary is at least alpha times the query in every coordinate.
Alpha=1 preserves the actual source amplitudes. The separate
support-only baseline requires positive activation and is exactly
a classical binary-context query. A zero meet retrieves all rows.

Six source rows are refreshed in the working matrix to remove small
numerical inconsistencies between freshly computed token codes and
cached sequence maxima. The other 25,594 rows are unchanged.
Original caches are never overwritten. The reports record every
refresh, input SHA-256 hashes, software versions, and code hash.

Counts include the source; subtract one for other retrieved rows.
The lexical reference uses explicit case-insensitive phrase or
whole-word rules. Lexical coincidence is not semantic correctness.
Statistics describe this fixed corpus and these selected examples;
there is no random-sample or held-out generalization claim.

In [2]:
pd.DataFrame([
    dict(case=name, source_row=row, positions=positions,
         lexical_patterns=patterns)
    for name, (row, positions, patterns) in CASES.items()
])

,case,source_row,positions,lexical_patterns
0,nyc,3457,"[1, 2, 3]",[\bnew\s+york\s+city\b]
1,animals,4042,"[8, 82]","[\bcats?\b, \bdogs?\b]"
2,rio,5411,"[15, 16, 17]",[\brio\s+de\s+janeiro\b]
3,richmond,117,"[30, 31]",[\brichmond\s+hill\b]
4,park,21481,"[24, 25, 26]",[\bnational\s+park\s+service\b]
5,sports,1924,"[15, 39, 94]","[\bcleveland\b, \bclippers\b, \bcavaliers\b]"


## Layer 0: execution and statistics

Execute the full protocol for this layer. The returned rows
cover full-vector meets, joins, and the Rio/de pair. Alpha
`None` (displayed as NaN) denotes the support-only
baseline, never alpha zero.
Closure is computed from all sequence summaries at levels
0.5 and 1; its extent is checked against the original extent.
Each extent, query, and checked closed intent is saved in NPZ.

In [3]:
report_0 = run('sae', 0)
frame_0 = pd.json_normalize(report_0['records'])
frame_0[[
    'case', 'operation', 'alpha', 'active_coordinates',
    'count', 'non_source_count', 'retained_support',
    'lexical.both', 'lexical_hit_fraction',
    'lexical_coverage', 'closure_added_coordinates',
]]

Matplotlib is building the font cache; this may take a moment.


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

Loaded pretrained model gpt2-small into HookedTransformer


/Users/ltsinadze/git/my_papers/.venv/lib/python3.14/site-packages/sae_lens/saes/sae.py:254: UserWarning: 
This SAE has non-empty model_from_pretrained_kwargs. 
For optimal performance, load the model like so:
model = HookedSAETransformer.from_pretrained_no_processing(..., **cfg.model_from_pretrained_kwargs)
  warnings.warn(


Moving model to device:  cpu
Moving model to device:  cpu


...

sae 0 park join 21 [4, 3, 1, 1, 1]


sae 0 sports meet 0 [25600, 25600, 25600, 25600, 25600]


sae 0 sports join 92 [1, 1, 1, 1, 1]


Saved sae_layer0 65 queries


,case,operation,alpha,active_coordinates,count,non_source_count,retained_support,lexical.both,lexical_hit_fraction,lexical_coverage,closure_added_coordinates
0,nyc,meet,NaN,1,25600,25599,1.000000,55,0.002148,1.000000,NaN
1,nyc,meet,0.25,1,25596,25595,0.999844,55,0.002149,1.000000,NaN
2,nyc,meet,0.50,1,25113,25112,0.980977,53,0.002110,0.963636,41.0
3,nyc,meet,0.75,1,24359,24358,0.951523,51,0.002094,0.927273,NaN
4,nyc,meet,1.00,1,21569,21568,0.842539,44,0.002040,0.800000,42.0
...,...,...,...,...,...,...,...,...,...,...,...
60,sports,join,NaN,92,1,0,1.000000,1,1.000000,1.000000,NaN
61,sports,join,0.25,92,1,0,1.000000,1,1.000000,1.000000,NaN
62,sports,join,0.50,92,1,0,1.000000,1,1.000000,1.000000,284.0
63,sports,join,0.75,92,1,0,1.000000,1,1.000000,1.000000,NaN


## Layer 8: execution and statistics

Execute the full protocol for this layer. The returned rows
cover full-vector meets, joins, and the Rio/de pair. Alpha
`None` (displayed as NaN) denotes the support-only
baseline, never alpha zero.
Closure is computed from all sequence summaries at levels
0.5 and 1; its extent is checked against the original extent.
Each extent, query, and checked closed intent is saved in NPZ.

In [4]:
report_8 = run('sae', 8)
frame_8 = pd.json_normalize(report_8['records'])
frame_8[[
    'case', 'operation', 'alpha', 'active_coordinates',
    'count', 'non_source_count', 'retained_support',
    'lexical.both', 'lexical_hit_fraction',
    'lexical_coverage', 'closure_added_coordinates',
]]

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

Loaded pretrained model gpt2-small into HookedTransformer
Moving model to device:  cpu
Moving model to device:  cpu


/Users/ltsinadze/git/my_papers/.venv/lib/python3.14/site-packages/sae_lens/saes/sae.py:254: UserWarning: 
This SAE has non-empty model_from_pretrained_kwargs. 
For optimal performance, load the model like so:
model = HookedSAETransformer.from_pretrained_no_processing(..., **cfg.model_from_pretrained_kwargs)
  warnings.warn(


nyc 3457 [1, 2, 3] [' New', ' York', ' City']


animals 4042 [8, 82] [' Cat', ' dog']


...

sae 8 park join 155 [1, 1, 1, 1, 1]


sae 8 sports meet 3 [546, 395, 288, 213, 148]


sae 8 sports join 175 [1, 1, 1, 1, 1]


Saved sae_layer8 65 queries


,case,operation,alpha,active_coordinates,count,non_source_count,retained_support,lexical.both,lexical_hit_fraction,lexical_coverage,closure_added_coordinates
0,nyc,meet,NaN,1,7421,7420,1.000000,13,0.001752,0.236364,NaN
1,nyc,meet,0.25,1,6619,6618,0.891928,11,0.001662,0.200000,NaN
2,nyc,meet,0.50,1,5956,5955,0.802587,11,0.001847,0.200000,8.0
3,nyc,meet,0.75,1,5451,5450,0.734537,9,0.001651,0.163636,NaN
4,nyc,meet,1.00,1,4935,4934,0.665005,9,0.001824,0.163636,8.0
...,...,...,...,...,...,...,...,...,...,...,...
60,sports,join,NaN,175,1,0,1.000000,1,1.000000,1.000000,NaN
61,sports,join,0.25,175,1,0,1.000000,1,1.000000,1.000000,NaN
62,sports,join,0.50,175,1,0,1.000000,1,1.000000,1.000000,2919.0
63,sports,join,0.75,175,1,0,1.000000,1,1.000000,1.000000,NaN


## Layer 11: execution and statistics

Execute the full protocol for this layer. The returned rows
cover full-vector meets, joins, and the Rio/de pair. Alpha
`None` (displayed as NaN) denotes the support-only
baseline, never alpha zero.
Closure is computed from all sequence summaries at levels
0.5 and 1; its extent is checked against the original extent.
Each extent, query, and checked closed intent is saved in NPZ.

In [5]:
report_11 = run('sae', 11)
frame_11 = pd.json_normalize(report_11['records'])
frame_11[[
    'case', 'operation', 'alpha', 'active_coordinates',
    'count', 'non_source_count', 'retained_support',
    'lexical.both', 'lexical_hit_fraction',
    'lexical_coverage', 'closure_added_coordinates',
]]

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

Loaded pretrained model gpt2-small into HookedTransformer
Moving model to device:  cpu
Moving model to device:  cpu


/Users/ltsinadze/git/my_papers/.venv/lib/python3.14/site-packages/sae_lens/saes/sae.py:254: UserWarning: 
This SAE has non-empty model_from_pretrained_kwargs. 
For optimal performance, load the model like so:
model = HookedSAETransformer.from_pretrained_no_processing(..., **cfg.model_from_pretrained_kwargs)
  warnings.warn(


nyc 3457 [1, 2, 3] [' New', ' York', ' City']


animals 4042 [8, 82] [' Cat', ' dog']


...

sae 11 park join 136 [1, 1, 1, 1, 1]


sae 11 sports meet 3 [231, 142, 79, 46, 21]


sae 11 sports join 186 [1, 1, 1, 1, 1]


Saved sae_layer11 65 queries


,case,operation,alpha,active_coordinates,count,non_source_count,retained_support,lexical.both,lexical_hit_fraction,lexical_coverage,closure_added_coordinates
0,nyc,meet,NaN,4,2627,2626,1.000000,10,0.003807,0.181818,NaN
1,nyc,meet,0.25,4,2136,2135,0.813095,9,0.004213,0.163636,NaN
2,nyc,meet,0.50,4,1722,1721,0.655501,7,0.004065,0.127273,10.0
3,nyc,meet,0.75,4,1323,1322,0.503616,7,0.005291,0.127273,NaN
4,nyc,meet,1.00,4,925,924,0.352113,4,0.004324,0.072727,10.0
...,...,...,...,...,...,...,...,...,...,...,...
60,sports,join,NaN,186,1,0,1.000000,1,1.000000,1.000000,NaN
61,sports,join,0.25,186,1,0,1.000000,1,1.000000,1.000000,NaN
62,sports,join,0.50,186,1,0,1.000000,1,1.000000,1.000000,2570.0
63,sports,join,0.75,186,1,0,1.000000,1,1.000000,1.000000,NaN


## Reading the outcomes

A shrinking extent across positive alpha levels is evidence that
magnitudes affect retrieval while the active-coordinate pattern
stays fixed. A constant extent does not demonstrate that effect.
A one-row extent retrieves only the source. A zero meet is a
degenerate query, not evidence for a shared semantic feature.

The manuscript compares these sets with the other surrogate using
all four cells of a coincidence table and Jaccard overlap. Only
corpus row identities are compared across models; latent feature
indices and activation units are not assumed to be aligned.

In [6]:
all_records = pd.concat([frame_0, frame_8, frame_11], ignore_index=True)
print('Evaluated extents:', len(all_records))
print('All closure and lattice-identity assertions passed.')
print('Results directory:', OUT)

Evaluated extents: 195
All closure and lattice-identity assertions passed.
Results directory: /Users/ltsinadze/git/my_papers/texs/sparsesurrs/lattcontexts/experiments/activation_results
